# Assignment


## Brief

Write the Python codes for the following questions.


## Instructions


Paste the answer as Python in the answer code section below each question.


### Question 1

Join the `metadata_pl` and `ratings_pl` DataFrames in Polars, then calculate the average rating by `original_language` (separately).


In [2]:
import requests
import os

# Download the files
gcs_bucket_url = "https://storage.googleapis.com/su-artifacts/"
file_names = ["movies_metadata.csv", "ratings.csv"]

for name in file_names:
    if os.path.exists('../data/'+name):
        print (f"File: {name} already exist in your data folder. Skipping download.")
    else:
        full_url = gcs_bucket_url + name
        r = requests.get(full_url)
        with open("../data/" + name, "wb") as f:
            f.write(r.content)

File: movies_metadata.csv already exist in your data folder. Skipping download.
File: ratings.csv already exist in your data folder. Skipping download.


**Answer:**

In [3]:
import polars as pl

metadata_pl = pl.read_csv(
    "../data/movies_metadata.csv",
    infer_schema_length=100000
).select(["id", "original_language"])

ratings_pl = pl.read_csv(
    "../data/ratings.csv"
).select(["movieId", "rating"])

metadata_pl = metadata_pl.with_columns(
    pl.col("id").cast(pl.Utf8)
)

ratings_pl = ratings_pl.with_columns(
    pl.col("movieId").cast(pl.Utf8)
)

joined_df = metadata_pl.join(
    ratings_pl,
    left_on="id",
    right_on="movieId",
    how="inner"
)

avg_rating_by_language = (
    joined_df
    .group_by("original_language")
    .agg(
        pl.mean("rating").alias("avg_rating")
    )
    .sort("avg_rating", descending=True)
)

avg_rating_by_language

original_language,avg_rating
str,f64
"""et""",4.163213
"""ru""",3.914689
"""fa""",3.775551
"""hu""",3.751581
"""hi""",3.675194
…,…
"""ca""",2.532407
"""id""",2.463415
null,2.458333


In [4]:
type(joined_df)

polars.dataframe.frame.DataFrame

### Question 2

Calculate the average total amount for vendors with at least 5 trips from the NYC Taxi Trip Data.

In [5]:
gcs_bucket_url = "https://storage.googleapis.com/su-artifacts/"
full_url = gcs_bucket_url + 'taxi_trip_data.csv'

# Check if file already exists and in that case skip download, otherwise download
if os.path.exists('../data/taxi_trip_data.csv'):
    print("File already exist.")
else:
    print("Downloading taxi_trip_data.csv...")
    print("Please note that this is a very large file (~1.5GB). Please make sure you have enough disk space and a stable internet connection.")
    print("This may take several minutes...")
    r = requests.get(full_url)
    with open("../data/" + 'taxi_trip_data.csv', "wb") as f:
        f.write(r.content)

File already exist.


**Answer**

In [6]:
import polars as pl

result = (
    pl.scan_csv("../data/taxi_trip_data.csv", try_parse_dates=True)
    .group_by("vendor_id")
    .agg(
        pl.len().alias("trip_count"),
        pl.mean("total_amount").alias("avg_total_amount")
    )
    .filter(pl.col("trip_count") >= 5)
    .sort("vendor_id")
    .collect(engine="streaming")
)

result

vendor_id,trip_count,avg_total_amount
i64,u32,f64
1,3949867,39.786292
2,6003561,41.010683
4,46572,38.650614


In [7]:
print(result)

shape: (3, 3)
┌───────────┬────────────┬──────────────────┐
│ vendor_id ┆ trip_count ┆ avg_total_amount │
│ ---       ┆ ---        ┆ ---              │
│ i64       ┆ u32        ┆ f64              │
╞═══════════╪════════════╪══════════════════╡
│ 1         ┆ 3949867    ┆ 39.786292        │
│ 2         ┆ 6003561    ┆ 41.010683        │
│ 4         ┆ 46572      ┆ 38.650614        │
└───────────┴────────────┴──────────────────┘
